<div style="font-family:system-ui;border-left:6px solid #B8721A;padding:6px 16px">
<div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#566273">The AI-Ready Organisation · Lab 5 · Module 5 · ≈35 min</div>
<div style="font-size:26px;font-weight:800;margin:6px 0">Price the confusion matrix</div>
<div style="font-size:15px"><b>You will:</b> turn a fraud model's precision and recall into money, find the threshold that minimises total cost, and see why 'accuracy' misleads executives.</div>
<div style="font-size:13px;color:#566273;margin-top:6px">Dataset: Credit Card Fraud Detection (284,807 anonymised European card transactions, 0.17% fraud; public Kaggle dataset)</div>
</div>

**How to use this notebook** · No coding needed.
1. In the menu choose **Runtime → Run all** (in Kaggle: **Run All**). The first run takes 1–3 minutes.
2. Change values in the **form fields** on the right of each cell (sliders, dropdowns). The cell reruns when you change a value; if it does not, press the play button.
3. Read the plain-English findings under each chart and record your answers in the final cell.

If the real dataset cannot be downloaded (for example, without a Kaggle sign-in), the notebook automatically switches to a **realistic simulated copy with the same structure** and tells you so. The lessons still hold.

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import warnings, os, io, zipfile, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.figsize": (9, 4.4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 11, "axes.titleweight": "bold"})
BUILD, BUY, BLEND, INK, GOOD, BAD = "#B8721A", "#13807B", "#4255B0", "#111822", "#2D7A4C", "#B03A3A"
pd.set_option("display.max_columns", 30)

def say(text):
    display(Markdown(text))

def scorecard(title, rows):
    """rows: list of (label, value, note)"""
    body = "".join(f"<tr><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px;font-weight:700;text-align:right'>{b}</td>"
                   f"<td style='padding:6px 12px;color:#566273'>{c}</td></tr>" for a, b, c in rows)
    display(HTML(f"<div style='font-family:system-ui;margin:8px 0'><div style='font-weight:700;font-size:15px;margin-bottom:4px'>{title}</div>"
                 f"<table style='border-collapse:collapse;font-size:14px'>{body}</table></div>"))

def try_kagglehub(handle, filename):
    """Download a public Kaggle dataset. Returns a file path or None."""
    try:
        try:
            import kagglehub
        except ImportError:
            os.system("pip -q install kagglehub")
            import kagglehub
        path = kagglehub.dataset_download(handle)
        for root, _, files in os.walk(path):
            if filename in files:
                return os.path.join(root, filename)
    except Exception as e:
        print(f"Kaggle download unavailable ({type(e).__name__}).")
    return None

RNG = np.random.default_rng(42)
DATA_NOTE = ""

## 1 · Load the transactions
Features V1–V28 are anonymised by the data provider; you will not need to interpret them.

In [ ]:
#@title Load data { display-mode: "form" }
p = try_kagglehub("mlg-ulb/creditcardfraud", "creditcard.csv")
df = None
if p:
    df = pd.read_csv(p); DATA_NOTE = "Real credit card fraud data (Kaggle)."
else:
    try:
        from sklearn.datasets import fetch_openml
        df = fetch_openml(data_id=1597, as_frame=True, parser="auto").frame
        df["Class"] = df["Class"].astype(int); DATA_NOTE = "Real credit card fraud data (OpenML mirror)."
    except Exception:
        from sklearn.datasets import make_classification
        X, y = make_classification(n_samples=284807, n_features=28, n_informative=10, weights=[.99827], class_sep=2.0, flip_y=0, random_state=0)
        df = pd.DataFrame(X, columns=[f"V{i}" for i in range(1, 29)]); df["Amount"] = RNG.gamma(1.2, 70, len(df)); df["Class"] = y
        DATA_NOTE = "Simulated copy (downloads unavailable). Same shape and fraud rate as the real dataset."
say(f"**{DATA_NOTE}** {len(df):,} transactions, **{df.Class.sum():,} fraudulent ({df.Class.mean():.2%})**.")
say(f"A model that says *'never fraud'* would be **{1-df.Class.mean():.2%} accurate** and catch nothing. That is why we price errors instead.")

In [ ]:
#@title Train the fraud model { display-mode: "form" }
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
feats = [c for c in df.columns if c.startswith("V")] + ["Amount"]
Xtr, Xte, ytr, yte = train_test_split(df[feats], df.Class, test_size=.3, random_state=1, stratify=df.Class)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ytr)
score = model.predict_proba(Xte)[:, 1]; yv = yte.to_numpy()
say(f"Model trained on {len(Xtr):,} transactions and tested on {len(Xte):,} it has never seen.")

## 2 · Set your threshold and your costs
Use your organisation's own figures: the average loss when fraud gets through, and the cost of an analyst reviewing one alert (plus customer friction).

In [ ]:
#@title Price the errors { display-mode: "form" }
threshold = 0.9 #@param {type:"slider", min:0.05, max:0.99, step:0.01}
loss_per_missed_fraud = 250 #@param {type:"number"}
cost_per_alert = 6 #@param {type:"number"}
def outcome(t):
    f = score >= t; tp = int((f & (yv == 1)).sum()); fp = int((f & (yv == 0)).sum()); fn = int(((~f) & (yv == 1)).sum()); tn = len(yv) - tp - fp - fn
    return tp, fp, fn, tn, fn*loss_per_missed_fraud + (tp+fp)*cost_per_alert
tp, fp, fn, tn, cost = outcome(threshold)
per100k = 1e5/len(yv)
fig, ax = plt.subplots(figsize=(6, 4))
cm = np.array([[tn, fp], [fn, tp]]); money = np.array([[0, fp*cost_per_alert], [fn*loss_per_missed_fraud, tp*cost_per_alert]])
ax.imshow(np.array([[0, 1], [2, 0.5]]), cmap="RdYlGn_r", alpha=.35)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}\n${money[i,j]:,.0f}", ha="center", va="center", fontsize=12, fontweight="bold")
ax.set_xticks([0, 1], ["Cleared", "Flagged"]); ax.set_yticks([0, 1], ["Genuine", "Fraud"]); ax.grid(False)
ax.set_title("Confusion matrix, priced"); plt.show()
scorecard(f"Threshold {threshold:.2f} on {len(yv):,} test transactions", [
    ("Recall (fraud caught)", f"{tp/(tp+fn):.0%}", f"{tp} of {tp+fn}"),
    ("Precision (alerts that are fraud)", f"{tp/max(tp+fp,1):.0%}", f"{fp:,} false alarms"),
    ("Accuracy", f"{(tp+tn)/len(yv):.2%}", "looks great, says little"),
    ("Total cost", f"${cost:,.0f}", f"≈ ${cost*per100k:,.0f} per 100,000 transactions")])

## 3 · Find the cheapest threshold

In [ ]:
#@title Cost curve { display-mode: "form" }
ts = np.round(np.r_[np.arange(.05, .9, .01), np.arange(.9, .9995, .002)], 3); res = np.array([outcome(t) for t in ts])
costs, acc = res[:, 4], (res[:, 0] + res[:, 3]) / len(yv)
t_cost, t_acc = ts[costs.argmin()], ts[acc.argmax()]
fig, ax = plt.subplots()
ax.plot(ts, costs/1e3, color=BUILD, lw=2.5); ax.fill_between(ts, costs/1e3, color=BUILD, alpha=.1)
ax.axvline(t_cost, color=GOOD, ls="--"); ax.text(t_cost, costs.max()/1e3*.95, f" lowest cost {t_cost:.3f}", color=GOOD)
ax.axvline(t_acc, color=BLEND, ls=":"); ax.text(t_acc, costs.max()/1e3*.8, f" highest accuracy {t_acc:.3f}", color=BLEND, ha="right")
ax.scatter([threshold], [cost/1e3], s=80, color=INK, zorder=5)
ax.set(title="Total cost by threshold", xlabel="Threshold", ylabel="Cost ($k, test set)"); plt.show()
c_acc = costs[acc.argmax()]
say(f"The cost-minimising threshold is **{t_cost:.3f}** (cost ${costs.min():,.0f}). The accuracy-maximising threshold is **{t_acc:.3f}** (cost ${c_acc:,.0f}).  \n"
    f"Optimising for accuracy would cost **${c_acc - costs.min():,.0f} more** on this test set. Change the cost fields above and rerun: the best threshold moves with your economics, not with the model.")

## 4 · Write the KPI chain
| Model quality | Operational | Business outcome | Financial |
|---|---|---|---|
| Recall, precision | Alerts per day, review time | Fraud losses avoided, customer complaints | Net loss rate, operations cost |

Now write the same four-link chain for your own capstone use case below.

## Your reflection → capstone §5
Answer in the form fields, then run the cell. Copy the summary into §5 of your business case on the course page.

In [ ]:
#@title Record your answers { display-mode: "form" }
q1 = "" #@param {type:"string"}
q2 = "" #@param {type:"string"}
q3 = "" #@param {type:"string"}
answers = [("Cost-minimising threshold with my organisation's costs, and why it differs from the accuracy-maximising one:", q1), ("My capstone KPI chain (model → operational → business → financial):", q2), ("Which leading indicator I will report in the first 30 days:", q3)]
say("### Lab 5 reflection")
for q, a in answers:
    say(f"**{q}**  \n{a if a.strip() else '_(not answered yet)_'}")